<a href="https://colab.research.google.com/github/joseportocarrero-stack/DataScience-Homework/blob/main/CanvasProject/CanvasAutomationTaskProject_v1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The purpose of this script is the automation of homework and exam search for my courses enrolled in Tecsup.

In [1]:
import os
from canvasapi import Canvas
from datetime import datetime, timezone
import zoneinfo

# Read secrets from environment
API_URL = os.getenv("CANVAS_URL")
API_KEY = os.getenv("CANVAS_KEY")

canvas = Canvas(API_URL, API_KEY)
LOCAL_TZ = zoneinfo.ZoneInfo("America/Lima")   # GMT-5

ModuleNotFoundError: No module named 'canvasapi'

In [ ]:
def get_tasks():
    print("Conecting a Canvas...\n")
    user = canvas.get_current_user()
    courses = user.get_courses(enrollment_state="active")
    report = []

    now_local = datetime.now(LOCAL_TZ)
    start_of_today = now_local.replace(hour=0, minute=0, second=0, microsecond=0)

    for course in courses:
        # Old courses
        if not hasattr(course, 'name'):
            continue
        print(f"Analyzing: {course.name}...")

        # Obtaining tasks
        try:
            tasks = course.get_assignments()
            for task in tasks:
              if task.due_at:
                # Parse UTC timestamp and convert to local time
                deadline_utc = datetime.strptime(
                    task.due_at, "%Y-%m-%dT%H:%M:%SZ"
                    ).replace(tzinfo=timezone.utc)
                    deadline_local = deadline_utc.astimezone(LOCAL_TZ)
                    # Filter: Only show tasks that are due as of today
                    if deadline > datetime.utcnow():
                        report.append({
                            "Course": course.name,
                            "Type": "Task",
                            "Name": task.name,
                            "Deadline": deadline.strftime("%Y-%m-%d %H:%M"),
                            "Link": task.html_url
                        })
        except Exception as e:
            # Sometimes there are no permissions or the course does not have the module active
            pass

    print("\n" + "="*50)
    print(" REPORT OF PENDING ASSIGNMENTS AND EXAMS ")
    print("="*50)

    if not report:
        print("Up to date! No pending tasks.")
    else:
        for item in report:
            print(f"\n📚 [{item['Course']}]")
            print(f"📌 {item['Type']}: {item['Name']}")
            print(f"📅 Deadline: {item['Deadline']}")
            print(f"🔗 Link: {item['Link']}")
            print("-" * 30)

In [ ]:
if __name__ == "__main__":
  get_tasks()